
# Agentic Multimodal Demo (Notebook)
Lightweight, agentic pipeline you can run step-by-step:
- **SeriesBuilder**: ordered people/things with dates (kings, CEOs, etc.) → SDXL portraits → poster
- **MapBuilder**: regions/points (Europe, US states, Canadian provinces, South America) → flags/icons → map

Repo layout assumption:
```
/notebooks/agentic_multimodal/agentic-multimodal.ipynb      # this notebook (root)
/src/agentic_multimodal #supporting modules
/results/agentic_multimodal    # output data
```


In [1]:

# --- Optional installs (uncomment as needed) ---
# %pip install langgraph langchain pydantic diffusers transformers accelerate
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install pillow geopandas shapely requests matplotlib bs4 pyproj
# %pip install ipywidgets
# Note: On Macs, torch install line will differ (MPS). See PyTorch docs.


In [2]:
import asyncio, uuid, json, pathlib as p
from agentic_multimodal.notebook_utils import make_registry, run_flow, tail_manifest, show_image

ROOT = p.Path("..").resolve().parent
REG = make_registry(ROOT)
MANIFEST = ROOT / "results" / "agentic_multimodal" / "manifest.jsonl"

# choose your prompt to show routing:
user_prompt = "make me a map of Europe with each country's flag at its capital"  # or presidents poster

result = asyncio.run(run_flow(user_prompt, REG))
print(result["kind"], "→ run_id:", result["run_id"])

# display
out_path = p.Path(result["output_path"])
show_image(out_path)

# peek provenance
tail_manifest(MANIFEST, n=1)


RuntimeError: Missing expected module. Ensure your package layout matches the suggested
structure and that you've `pip install -e .` the repo.
Original import error: cannot import name 'WikidataSeries' from 'agentic_multimodal.skills.data.wikidata_series' (/Users/douglasdaly/Documents/GitHub/Generative-AI/src/agentic_multimodal/skills/data/wikidata_series.py)

In [2]:
from pathlib import Path
import sys
from assets.support import tree_markdown
from assets.agentic_multimodal.src.agents.image_gen import get_sdxl

from pathlib import Path; import sys
SRC = Path.cwd().resolve()/"assets"/"agentic_multimodal"/"src" 
sys.path.insert(0, str(SRC))
from core.config import CACHE, RESULTS

for dir in [CACHE, RESULTS]:
    dir.mkdir(parents=True, exist_ok=True)  # ensure directory exists
    if str(dir) not in sys.path:
        sys.path.insert(0, str(dir))

# Show code tree from source:
display(tree_markdown(SRC))

/opt/anaconda3/envs/ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


```src/
├── adapters
│   ├── __init__.py
│   ├── natural_earth.py
│   ├── wikidata_geo.py
│   └── wikidata_series.py
├── agents
│   ├── __init__.py
│   ├── compositor.py
│   ├── image_gen.py
│   └── research.py
├── core
│   ├── __init__.py
│   └── config.py
├── tasks
│   └── __init__.py
├── tools
│   ├── __init__.py
│   ├── render_map.py
│   ├── render_poster.py
│   ├── research_people.py
│   └── web.py
├── __init__.py
├── graph.py
└── schemas.py
```

## SeriesBuilder: Wikidata adapters (leaders, CEOs, etc.)

## Image generation (SDXL via Diffusers)

## Poster compositor (grid + captions)

## MapBuilder: regions + capitals + flags

In [6]:

from assets.agentic_multimodal.src.adapters.wikidata_geo import get_points_for_region, render_region_map

#base, pts, label = get_points_for_region("Europe")
#out = render_region_map(base, pts, outpath="out/europe_flags.png", title=f"{label}: Capitals & Flags")


### Demo: Series poster — POTUS, British monarchs, Nobel Prize winners
#### Collect data

In [14]:
# POTUS
from assets.agentic_multimodal.src.adapters.wikidata_series import (
    series_monarchs_eng_gb_uk, series_nobel, series_potus
)
series = series_potus()


In [15]:
# Generate + compose with centered captions
paths = batch_generate_portraits(series, outdir="results/agentic_multimodal/potus", steps=20, scale=6.0)
poster = compose_poster(
    series, paths,
    cols=8,
    size=(768,1024),
    base_font_path=None,   
    base_font_px=26,
    caption_scale=2.0,
    outpath="results/agentic_multimodal/potus_768_1024.png",
    max_long_side=4096,
)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

[compose] canvas 6392 x 7040 (~45.0 MP)
[compose] downscaled to (3718, 4096) (~15.2 MP)


In [16]:

# British monarchs (full stretch)
series = series_monarchs_eng_gb_uk()
# Generate + compose with centered captions
paths = batch_generate_portraits(series, outdir="results/agentic_multimodal/monarchs", steps=20, scale=6.0)
poster = compose_poster(
    series, paths,
    cols=8,
    size=(768,1024),
    base_font_path=None,   
    base_font_px=26,
    caption_scale=2.0,
    outpath="results/agentic_multimodal/monarchs_england_great_britain_768_1024.png",
)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

[compose] canvas 6392 x 10532 (~67.3 MP)


In [ ]:
# Nobel prize winners in Physics

series = series_nobel('Q38104', 'Physics')
# Generate + compose with centered captions
paths = batch_generate_portraits(series, outdir="results/agentic_multimodal/physics_nobel", steps=20, scale=6.0)
poster = compose_poster(
    series, paths,
    cols=12,
    size=(768,1024),
    base_font_path=None,   
    base_font_px=26,
    caption_scale=2.0,
    outpath="results/agentic_multimodal/physics_nobel_768_1024.png",
)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:

if 0:
    # 2) Or, Nobel laureates in Physics
    #    (Q38104 = Nobel Prize in Physics)
    physics = series_nobel("Q38104", "Physics")

    # Render either with your existing pipeline:
    paths = batch_generate_portraits(series, outdir="results/monarchs", steps=20, scale=6.0)
    poster = compose_poster(series, paths, cols=12, size=(768,1024),
                            base_font_path=None, base_font_px=26, caption_scale=2.0,
                            outpath="results/monarchs_poster.png")


### Demo: Series poster — CEOs of General Electric

In [ ]:

# Resolve "General Electric" QID
wd_search_label("General Electric")[:3]


In [ ]:

GE_QID = "Q7729"  # verify with search above
series_ge = series_ceos_of_company(GE_QID, title="CEOs of GE (alpha template)")
len(series_ge["items"]), series_ge["items"][:3]


In [ ]:

paths_ge = batch_generate_portraits(series_ge, outdir="out/ge_ceos", steps=18, scale=6.0)
poster_ge = compose_poster(series_ge, paths_ge, cols=5, outpath="out/ge_ceos_poster.png")
poster_ge


### Demo: Map — Europe with capitals and flags

In [ ]:

world = load_admin0()
europe = filter_continent(world, "Europe")
df_caps = wikidata_capitals_for_continent("Q46")  # Europe
out_map = render_region_map(europe, df_caps, outpath="out/europe_map.png", epsg=3857)
out_map


### Demo: Map — South America

In [ ]:

south_america = filter_continent(world, "South America")
df_caps_sa = wikidata_capitals_for_continent("Q18")  # South America
out_map_sa = render_region_map(south_america, df_caps_sa, outpath="out/south_america_map.png", epsg=3857)
out_map_sa


### Demo: Map — US states (admin-1 centroids)

In [ ]:

admin1 = load_admin1()
us_states = filter_admin1_by_country(admin1, "United States")
# Compute approximate state centroids and label them; subnational flags need a different source (P41), so skip flags here.
centroids = us_states.to_crs(4326).copy()
centroids["lon"] = centroids.geometry.centroid.x
centroids["lat"] = centroids.geometry.centroid.y
pts = centroids[["name","lat","lon"]].rename(columns={"name":"country"})
pts["capital"] = ""  # placeholder
pts["iso2"] = ""     # no flags for subnationals via FlagCDN
out_us = render_region_map(us_states, pts, outpath="out/us_states_map.png", epsg=5070)  # Albers USA
out_us


## Re-render your `src` tree after you add files

In [ ]:

md = tree_markdown("./assets/agentic-multimodal/src")
display(Markdown(md))


# North Star

## Goal (1 sentence)
<What this system must do. No ANDs.>

## Non-Goals
- <Thing we are NOT solving>
- <Another non-goal>

## Primary Users & Usage
- **Who:** <persona/role>
- **How used:** <CLI/API/UI, frequency>
- **Success signal:** <what “done right” looks like>

## Public API Surface (contracts only)
- **Module/Service A**
  - `fn(sig) -> type` — <brief contract/invariants>
- **Module/Service B**
  - `endpoint METHOD /path` — <request/response shape>

## Invariants (must always hold)
- <e.g., “clockwise is UR→DR→DL→UL”>
- <e.g., “idempotent writes,” “no network in hot path,” “inputs validated at edge”>

## Performance/Scale Targets
- P50/P95 latency: <X ms>
- Memory cap: <Y MB>
- Throughput: <Z ops/sec>
- Max data size: <N>

## Extensibility Points
- <Where new features plug in—hooks, interfaces, events>

## Test Strategy
- Acceptance tests: <goldens you’ll keep stable>
- Contract tests: <per public interface>
- Property tests (if any): <invariants>

## Observability
- Log schema: `{event, component, id, elapsed_ms, ...}`
- Trace/Correlation: <how you propagate IDs>
- Metrics: <what you emit>

## Risks & Open Questions
- <Biggest unknowns to resolve>


# Decision: <short>
## Context
- <why change>
## Options
- A: <one line>
- B: <one line>
## Decision
- <chosen option>
## Consequences
- + <pros>
- − <cons>
## Migration/Notes
- <follow-ups, flags, rollbacks>


In [21]:
name = "this and that"
name = name.split(" and ", 1)[0].strip()
print(name)

this
